# CausalX Full Pipeline (Embeddings + CFN)

Minimal, end-to-end workflow: setup, cache wav2vec2, build embedding CSV, train CFN with embeddings, optional VisualTCN, inference demo, and balanced evaluation.

In [2]:
# 0) Environment & paths (works locally or in Colab)
from pathlib import Path
import sys, os, random, numpy as np, pandas as pd, librosa

LOCAL_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')
COLAB_ROOT = Path('/content/CausalX-Project/backend')
PROJECT_ROOT = LOCAL_ROOT if LOCAL_ROOT.exists() else COLAB_ROOT
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_ROOT = DATA_ROOT / 'raw' / 'fakeavceleb'
sys.path.insert(0, str(PROJECT_ROOT))

# Env for embeddings + mediapipe
os.environ['CFN_USE_EMBEDDINGS'] = 'true'
os.environ['CFN_EMB_MODEL_PATH'] = str(PROJECT_ROOT/'models/cfn_emb.pth')
os.environ['CFN_W2V2_MODEL'] = 'WAV2VEC2_BASE'
os.environ['MEDIAPIPE_DISABLE_GPU'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
vis_tcn = PROJECT_ROOT/'models/visual_tcn.pth'
if vis_tcn.exists():
    os.environ['CFN_VISUAL_TCN_PATH'] = str(vis_tcn)
    print('Using VisualTCN:', vis_tcn)
else:
    os.environ['CFN_VISUAL_TCN_PATH'] = ''
    print('VisualTCN not found; continuing without it')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_ROOT exists:', RAW_ROOT.exists())
print('CFN_EMB_MODEL_PATH:', os.environ['CFN_EMB_MODEL_PATH'])


Using VisualTCN: /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/models/visual_tcn.pth
PROJECT_ROOT: /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend
RAW_ROOT exists: True
CFN_EMB_MODEL_PATH: /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/models/cfn_emb.pth


In [ ]:
# 1) Cache wav2vec2 checkpoint (one-time)
import pathlib
ckpt_dir = pathlib.Path('~/.cache/torch/hub/checkpoints').expanduser()
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt = ckpt_dir/'wav2vec2_fairseq_base_ls960.pth'
if not ckpt.exists():
    !wget -O "{ckpt}" https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960.pth
print('wav2vec2 cached:', ckpt.exists(), ckpt)


In [ ]:
# 2) Build embedding-augmented CSV (full dataset)

from pathlib import Path
import os, random, pandas as pd, numpy as np, librosa
from src.utils.dataset_registry import get_fakeavceleb_videos
from src.cvi.feature_extractor import FeatureExtractor
from src.cvi.frame_causal_extractor import extract_frame_level_features

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
os.environ.setdefault('MEDIAPIPE_DISABLE_GPU', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')

fe = FeatureExtractor()
videos = get_fakeavceleb_videos(RAW_ROOT)
random.shuffle(videos)
rows = []
for v in videos:  # full set
    path = Path(v['path'])
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    try:
        frames = extract_frame_level_features(str(path), duration=8.0)
    except Exception as e:
        print('skip (frames):', path, e)
        continue
    if not frames:
        continue
    lip = np.array([f['lip_aperture'] for f in frames], np.float32)
    audio_rms = np.array([f['audio_rms'] for f in frames], np.float32)
    try:
        wav, sr = librosa.load(str(path), sr=16000, duration=8.0)
    except Exception as e:
        print('skip (audio):', path, e)
        continue
    vis = fe.get_visual_embeddings(lip)
    wav_emb = fe.get_audio_embeddings(wav, sr=sr)
    rows.append({
        'path': str(path),
        'label': v['label'],
        'lip_variance': float(lip.var()) if lip.size else 0.0,
        'av_correlation': float(np.corrcoef(lip, audio_rms)[0,1]) if len(lip)>1 else 0.0,
        'av_lag_frames': 0.0,
        'jitter_mean': float(np.mean([f.get('jitter',0.0) for f in frames])),
        'jitter_std': float(np.std([f.get('jitter',0.0) for f in frames])),
        'tcn_visual_emb': float(np.mean(vis)) if vis.size else 0.0,
        'wav2vec_audio_emb': float(np.mean(wav_emb)) if wav_emb.size else 0.0,
    })

out = DATA_ROOT / 'processed' / 'causal_multimodal_embeddings_full.csv'
out.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_csv(out, index=False)
print('wrote', len(rows), 'rows to', out)


In [ ]:
# 2b) Split full embeddings CSV into train/test
import pandas as pd
from sklearn.model_selection import train_test_split
full_csv = DATA_ROOT/'processed'/'causal_multimodal_embeddings_full.csv'
df = pd.read_csv(full_csv)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_path = DATA_ROOT/'processed'/'causal_multimodal_embeddings_train.csv'
test_path = DATA_ROOT/'processed'/'causal_multimodal_embeddings_test.csv'
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print('train', len(train_df), 'test', len(test_df))


In [ ]:
%%bash
cd /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend
source .venv/bin/activate
export PYTHONPATH=.
export CFN_USE_EMBEDDINGS=true
export CFN_W2V2_MODEL=WAV2VEC2_BASE
export CFN_EMB_MODEL_PATH=$(pwd)/models/cfn_emb.pth
export CFN_VISUAL_TCN_PATH=$(pwd)/models/visual_tcn.pth
export MEDIAPIPE_DISABLE_GPU=1
export CUDA_VISIBLE_DEVICES=-1
export MPLCONFIGDIR=/tmp/mpl
CAUSAL_WEIGHT=0.1  # tune 0.05–0.5

python -m src.training.train_cfn \
  --data data/processed/causal_multimodal_embeddings_train.csv \
  --use-scaler --use-embeddings \
  --epochs 25 --patience 6 --batch-size 128 \
  --causal-weight ${CAUSAL_WEIGHT}


In [ ]:
# 4) (Optional) Train a lightweight VisualTCN
from pathlib import Path
import sys, os
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')
sys.path.insert(0, str(PROJECT_ROOT))
from src.modules.embeddings import VisualTCN
from src.cvi.frame_causal_extractor import extract_frame_level_features
from src.utils.dataset_registry import get_fakeavceleb_videos
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import pandas as pd

TRAIN_CSV = PROJECT_ROOT/'data/processed/causal_multimodal_embeddings_train.csv'
if TRAIN_CSV.exists():
    df = pd.read_csv(TRAIN_CSV)
    videos = df.to_dict('records')
else:
    videos = get_fakeavceleb_videos(PROJECT_ROOT/'data/raw/fakeavceleb')

seqs, labels = [], []
for v in videos[:200]:
    path = Path(v['path'])
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    label = v['label']
    frames = extract_frame_level_features(str(path), duration=6.0)
    if len(frames) < 20:
        continue
    lip = np.array([f['lip_aperture'] for f in frames], np.float32)
    lip = (lip - lip.mean()) / (lip.std()+1e-6)
    seqs.append(lip[:256])
    labels.append(label)

if seqs:
    max_len = max(len(s) for s in seqs)
    seqs = [np.pad(s, (0, max_len-len(s))) for s in seqs]
    X = torch.tensor(seqs).float()
    y = torch.tensor(labels, dtype=torch.float32)
    model = VisualTCN(out_dim=64)
    clf = nn.Linear(64,1)
    opt = optim.Adam(list(model.parameters())+list(clf.parameters()), lr=1e-3)
    bce = nn.BCEWithLogitsLoss()
    for epoch in range(8):
        opt.zero_grad()
        z = model(X)
        loss = bce(clf(z).squeeze(1), y)
        loss.backward(); opt.step()
        print('epoch', epoch+1, 'loss', float(loss))
    (PROJECT_ROOT/'models').mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), PROJECT_ROOT/'models/visual_tcn.pth')
    print('saved visual TCN to', PROJECT_ROOT/'models/visual_tcn.pth')
else:
    print('No sequences found; skipping VisualTCN training')


In [ ]:
# 5) Inference demo
import os
from pathlib import Path
from src.cvi.api.inference_service import run_full_cvi_pipeline
PROJECT_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')
os.environ['CFN_USE_EMBEDDINGS'] = 'true'
os.environ['CFN_EMB_MODEL_PATH'] = str(PROJECT_ROOT/'models/cfn_emb.pth')
os.environ['CFN_W2V2_MODEL'] = 'WAV2VEC2_BASE'
os.environ['MEDIAPIPE_DISABLE_GPU'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
video_path = PROJECT_ROOT / 'data/raw/dfdc/train_sample_videos/emfbhytfhc.mp4'
try:
    r = run_full_cvi_pipeline(str(video_path))
    print('video_fake', r['video_fake'], 'confidence', r['fake_confidence'])
    print('highlights', r['highlight_timestamps'][:10])
except Exception as e:
    print('Inference failed for', video_path)
    print(e)


In [ ]:
# 6) Build balanced manifest (equal real/fake) for eval
import csv, random, pandas as pd
from pathlib import Path
from src.utils.dataset_registry import get_fakeavceleb_videos
# ensure roots
try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')
DATA_ROOT = PROJECT_ROOT/'data'
RAW_ROOT = DATA_ROOT/'raw'/'fakeavceleb'
TEST_CSV = DATA_ROOT/'processed'/'causal_multimodal_embeddings_test.csv'
if TEST_CSV.exists():
    df = pd.read_csv(TEST_CSV)
    videos = df.to_dict('records')
else:
    videos = get_fakeavceleb_videos(RAW_ROOT)
fakes = [v for v in videos if v['label']==1]
reals = [v for v in videos if v['label']==0]
n = min(len(fakes), len(reals), 200)
random.shuffle(fakes); random.shuffle(reals)
sample = fakes[:n] + reals[:n]
random.shuffle(sample)
manifest = PROJECT_ROOT/'eval_manifest_balanced.tsv'
with manifest.open('w') as f:
    w = csv.writer(f, delimiter='	')
    for v in sample:
        w.writerow([v['path'], v['label']])
print('Wrote', len(sample), 'rows to', manifest)


In [ ]:
# 6b) Downsample manifest for fast sweeps (set DS_N per class)
import random, csv
DS_N = 80
manifest_full = PROJECT_ROOT/'eval_manifest_balanced.tsv'
rows = manifest_full.read_text().splitlines()
random.shuffle(rows)
fakes = [r for r in rows if r.split('	')[1]=='1']
reals = [r for r in rows if r.split('	')[1]=='0']
ds = fakes[:DS_N] + reals[:DS_N]
random.shuffle(ds)
manifest_ds = PROJECT_ROOT/'eval_manifest_balanced_ds.tsv'
with manifest_ds.open('w') as f:
    for r in ds:
        f.write(r+'')
print('Downsampled manifest:', manifest_ds, 'rows', len(ds))


In [ ]:
import json, numpy as np, os
cache = json.load(open(PROJECT_ROOT/"eval_cache_ds.json"))

grid_prob  = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.85]
grid_ratio = [0.20, 0.30, 0.40, 0.50, 0.60]

results = []
for p in grid_prob:
    for r in grid_ratio:
        tp=tn=fp=fn=0
        for row in cache:
            # suspicious if both prob_mean>=p AND fake_conf>=r (matches current require_flag rule)
            pred = 1 if (row["prob_mean"] >= p and row["fake_conf"] >= r) else 0
            label = row["label"]
            if pred==1 and label==1: tp+=1
            elif pred==0 and label==0: tn+=1
            elif pred==1 and label==0: fp+=1
            else: fn+=1
        rec  = tp/(tp+fn) if (tp+fn) else 0
        spec = tn/(tn+fp) if (tn+fp) else 0
        bal  = 0.5*(rec+spec)
        results.append((bal,p,r,rec,spec,tp,tn,fp,fn))

best = max(results, key=lambda x: x[0])
bal,p,r,rec,spec,tp,tn,fp,fn = best
print(f"Best BalAcc={bal:.3f} at PROB={p:.2f} RATIO={r:.2f} | Rec={rec:.3f} Spec={spec:.3f} TP={tp} TN={tn} FP={fp} FN={fn}")


In [ ]:
# 8b) Cache scores on full balanced manifest (test split)
from pathlib import Path
import json, os
from src.cvi.api.inference_service import run_full_cvi_pipeline
os.environ['CFN_USE_EMBEDDINGS'] = 'true'
os.environ['CFN_EMB_MODEL_PATH'] = str(PROJECT_ROOT/'models/cfn_emb.pth')
manifest = PROJECT_ROOT/'eval_manifest_balanced.tsv'
rows = [ln for ln in manifest.read_text().splitlines() if ln.strip()]
print('rows:', len(rows))
cache = []
for line in rows:
    parts = line.split('	')
    if len(parts) < 2:
        continue
    path_str, label = parts[0], parts[1]
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    try:
        out = run_full_cvi_pipeline(str(p))
        cache.append({
            'path': str(p),
            'label': int(label),
            'fake_conf': out['fake_confidence'],
            'prob_mean': out['overall_score'],
            'breach': out.get('causal_breach_score', 0),
        })
    except Exception as e:
        print('Skip', p, e)
print('cached', len(cache))
with open(PROJECT_ROOT/'eval_cache_full.json', 'w') as f:
    json.dump(cache, f)


In [ ]:
# 8c) Sweep thresholds on full cache
import json
cache = json.load(open(PROJECT_ROOT/'eval_cache_full.json'))

grid_prob  = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
grid_ratio = [0.30, 0.40, 0.50, 0.60, 0.70]
results = []
for p in grid_prob:
    for r in grid_ratio:
        tp=tn=fp=fn=0
        for row in cache:
            pred = 1 if (row['prob_mean'] >= p and row['fake_conf'] >= r) else 0
            label = row['label']
            if pred==1 and label==1: tp+=1
            elif pred==0 and label==0: tn+=1
            elif pred==1 and label==0: fp+=1
            else: fn+=1
        rec  = tp/(tp+fn) if (tp+fn) else 0
        spec = tn/(tn+fp) if (tn+fp) else 0
        bal  = 0.5*(rec+spec)
        results.append((bal,p,r,rec,spec,tp,tn,fp,fn))

best = max(results, key=lambda x: x[0])
bal,p,r,rec,spec,tp,tn,fp,fn = best
print(f"Best BalAcc={bal:.3f} at PROB={p:.2f} RATIO={r:.2f} | Rec={rec:.3f} Spec={spec:.3f} TP={tp} TN={tn} FP={fp} FN={fn}")


In [ ]:
# 7b) Set thresholds from sweep 
import os
# Pair A: from sweep best
os.environ['CFN_PROB_THRESH'] = '0.50'
os.environ['CFN_RATIO_THRESH'] = '0.80'
# Pair B (optional stricter): uncomment to test
# os.environ['CFN_PROB_THRESH'] = '0.40'
# os.environ['CFN_RATIO_THRESH'] = '0.50'
os.environ['CFN_CAUSAL_THRESH'] = os.getenv('CFN_CAUSAL_THRESH','0.6')
print('Using thresholds', os.environ['CFN_PROB_THRESH'], os.environ['CFN_RATIO_THRESH'])


In [ ]:
# 7) Set thresholds and run balanced eval (full manifest)
import os
os.environ['CFN_PROB_THRESH'] = os.getenv('CFN_PROB_THRESH','0.5')
os.environ['CFN_RATIO_THRESH'] = os.getenv('CFN_RATIO_THRESH','0.8')
os.environ['CFN_CAUSAL_THRESH'] = os.getenv('CFN_CAUSAL_THRESH','0.25')
from src.cvi.api.inference_service import run_full_cvi_pipeline
from pathlib import Path
tp=tn=fp=fn=skipped=0
manifest = PROJECT_ROOT/'eval_manifest_balanced.tsv'
for line in manifest.read_text().splitlines():
    path_str, label = line.split('	')
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    try:
        r = run_full_cvi_pipeline(str(p))
    except Exception as e:
        skipped += 1
        print('Skip', p, e)
        continue
    pred = r['video_fake']; label=int(label)
    if pred==1 and label==1: tp+=1
    elif pred==0 and label==0: tn+=1
    elif pred==1 and label==0: fp+=1
    else: fn+=1
total = tp+tn+fp+fn
acc = (tp+tn)/total if total else 0
prec = tp/(tp+fp) if (tp+fp) else 0
rec = tp/(tp+fn) if (tp+fn) else 0
spec = tn/(tn+fp) if (tn+fp) else 0
bal_acc = 0.5*(rec+spec)
f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
print(f'Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} Spec={spec:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} TP={tp} TN={tn} FP={fp} FN={fn} Skipped={skipped}')


In [ ]:
# 7) Set thresholds and run balanced eval (full manifest)
import os, importlib
from pathlib import Path

os.environ["CFN_PROB_THRESH"] = "0.80"
os.environ["CFN_RATIO_THRESH"] = "0.60"
os.environ["CFN_CAUSAL_THRESH"] = "0.77"
os.environ["CFN_REQUIRE_FLAG"] = "true"   

# inference_service reads env at import time; force reload after changing env
import src.cvi.api.inference_service as inf
importlib.reload(inf)
run_full_cvi_pipeline = inf.run_full_cvi_pipeline

tp=tn=fp=fn=skipped=0
manifest = PROJECT_ROOT / "eval_manifest_balanced.tsv"

for line in manifest.read_text().splitlines():
    path_str, label = line.split("\t")
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    try:
        r = run_full_cvi_pipeline(str(p))
    except Exception as e:
        skipped += 1
        print("Skip", p, e)
        continue

    pred = r["video_fake"]
    label = int(label)
    if pred==1 and label==1: tp+=1
    elif pred==0 and label==0: tn+=1
    elif pred==1 and label==0: fp+=1
    else: fn+=1

total = tp+tn+fp+fn
acc = (tp+tn)/total if total else 0
prec = tp/(tp+fp) if (tp+fp) else 0
rec = tp/(tp+fn) if (tp+fn) else 0
spec = tn/(tn+fp) if (tn+fp) else 0
bal_acc = 0.5*(rec+spec)
f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
print(f"Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} Spec={spec:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} TP={tp} TN={tn} FP={fp} FN={fn} Skipped={skipped}")


In [ ]:
import numpy as np
from pathlib import Path
from src.cvi.api.inference_service import run_full_cvi_pipeline

manifest = PROJECT_ROOT / "eval_manifest_balanced.tsv"
rows = [ln for ln in manifest.read_text().splitlines() if ln.strip()]

# 1) Cache frame-level outputs once
cache = []
for ln in rows:
    path_str, label = ln.split("\t")
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    out = run_full_cvi_pipeline(str(p))
    frames = out["frames"]
    probs = np.array([f.get("fake_prob_smooth", f.get("fake_prob", 0.0)) for f in frames], dtype=float)
    mism  = np.array([f.get("av_mismatch", 0.0) for f in frames], dtype=float)
    cache.append({"label": int(label), "probs": probs, "mism": mism})

def eval_combo(prob_t, ratio_t, causal_t, require_flag):
    tp=tn=fp=fn=0
    for item in cache:
        y = item["label"]
        probs = item["probs"]
        mism = item["mism"]
        if len(probs) == 0:
            pred = 0
        else:
            if require_flag:
                suspicious = (probs >= prob_t) & (mism >= causal_t)   # AND
            else:
                suspicious = (probs >= prob_t) | (mism >= causal_t)   # OR
            ratio = suspicious.mean()
            pred = int(ratio >= ratio_t)

        if pred==1 and y==1: tp+=1
        elif pred==0 and y==0: tn+=1
        elif pred==1 and y==0: fp+=1
        else: fn+=1

    rec  = tp/(tp+fn) if (tp+fn) else 0
    spec = tn/(tn+fp) if (tn+fp) else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) else 0
    bal  = 0.5*(rec+spec)
    return (bal, f1, acc, prec, rec, spec, tp, tn, fp, fn)

grid_prob   = np.linspace(0.45, 0.75, 7)
grid_ratio  = np.linspace(0.25, 0.65, 9)
grid_causal = np.linspace(0.45, 0.75, 7)

best = None
for rf in [False, True]:  # OR then AND
    for p in grid_prob:
        for r in grid_ratio:
            for c in grid_causal:
                m = eval_combo(float(p), float(r), float(c), rf)
                # rank by BalAcc, then F1
                key = (m[0], m[1])
                if best is None or key > best["key"]:
                    best = {
                        "key": key, "require_flag": rf, "prob": p, "ratio": r, "causal": c,
                        "metrics": m
                    }

bal,f1,acc,prec,rec,spec,tp,tn,fp,fn = best["metrics"]
print(f"BEST require_flag={best['require_flag']} prob={best['prob']:.2f} ratio={best['ratio']:.2f} causal={best['causal']:.2f}")
print(f"Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} Spec={spec:.3f} BalAcc={bal:.3f} F1={f1:.3f} TP={tp} TN={tn} FP={fp} FN={fn}")


In [ ]:

p0, r0, c0 = best["prob"], best["ratio"], best["causal"]
rf0 = best["require_flag"]

grid_prob   = np.clip(np.arange(p0-0.05, p0+0.051, 0.01), 0, 1)
grid_ratio  = np.clip(np.arange(r0-0.10, r0+0.101, 0.02), 0, 1)
grid_causal = np.clip(np.arange(c0-0.05, c0+0.051, 0.01), 0, 1)

best2 = None
for p in grid_prob:
    for r in grid_ratio:
        for c in grid_causal:
            m = eval_combo(float(p), float(r), float(c), rf0)
            key = (m[0], m[1])  # BalAcc then F1
            if best2 is None or key > best2["key"]:
                best2 = {"key": key, "prob": p, "ratio": r, "causal": c, "metrics": m}

bal,f1,acc,prec,rec,spec,tp,tn,fp,fn = best2["metrics"]
print(f"REFINED prob={best2['prob']:.2f} ratio={best2['ratio']:.2f} causal={best2['causal']:.2f} rf={rf0}")
print(f"Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} Spec={spec:.3f} BalAcc={bal:.3f} F1={f1:.3f} TP={tp} TN={tn} FP={fp} FN={fn}")


## 10) Measure Current Deployed Accuracy
Use fixed runtime thresholds/model paths (no sweep) and evaluate on `eval_manifest_balanced.tsv`.


In [ ]:
# 10a) Runtime config for current deployed model
from pathlib import Path
import os, importlib

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')

os.environ['PYTHONPATH'] = '.'
os.environ['CFN_USE_EMBEDDINGS'] = 'true'
os.environ['CFN_EMB_MODEL_PATH'] = str(PROJECT_ROOT / 'models/cfn_emb.pth')
os.environ['CFN_VISUAL_TCN_PATH'] = str(PROJECT_ROOT / 'models/visual_tcn.pth')
os.environ['CFN_W2V2_MODEL'] = 'WAV2VEC2_BASE'
os.environ['CFN_PROB_THRESH'] = os.getenv('CFN_PROB_THRESH', '0.80')
os.environ['CFN_RATIO_THRESH'] = os.getenv('CFN_RATIO_THRESH', '0.60')
os.environ['CFN_CAUSAL_THRESH'] = os.getenv('CFN_CAUSAL_THRESH', '0.75')
os.environ['CFN_REQUIRE_FLAG'] = os.getenv('CFN_REQUIRE_FLAG', 'false')
os.environ['MEDIAPIPE_DISABLE_GPU'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['MPLCONFIGDIR'] = os.getenv('MPLCONFIGDIR', '/tmp/mpl')

import src.cvi.api.inference_service as inf
importlib.reload(inf)
run_full_cvi_pipeline = inf.run_full_cvi_pipeline

print('CFN_EMB_MODEL_PATH:', os.environ['CFN_EMB_MODEL_PATH'])
print('CFN_PROB_THRESH:', os.environ['CFN_PROB_THRESH'])
print('CFN_RATIO_THRESH:', os.environ['CFN_RATIO_THRESH'])
print('CFN_CAUSAL_THRESH:', os.environ['CFN_CAUSAL_THRESH'])
print('CFN_REQUIRE_FLAG:', os.environ['CFN_REQUIRE_FLAG'])


In [ ]:
# 10b) Evaluate current deployed behavior on balanced manifest
# 10a) Runtime config for current deployed model
from pathlib import Path
import os, importlib

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend')

os.environ['PYTHONPATH'] = '.'
os.environ['CFN_USE_EMBEDDINGS'] = 'true'
os.environ['CFN_EMB_MODEL_PATH'] = str(PROJECT_ROOT / 'models/cfn_emb.pth')
os.environ['CFN_VISUAL_TCN_PATH'] = str(PROJECT_ROOT / 'models/visual_tcn.pth')
os.environ['CFN_W2V2_MODEL'] = 'WAV2VEC2_BASE'
os.environ['CFN_PROB_THRESH'] = os.getenv('CFN_PROB_THRESH', '0.80')
os.environ['CFN_RATIO_THRESH'] = os.getenv('CFN_RATIO_THRESH', '0.60')
os.environ['CFN_CAUSAL_THRESH'] = os.getenv('CFN_CAUSAL_THRESH', '0.75')
os.environ['CFN_REQUIRE_FLAG'] = os.getenv('CFN_REQUIRE_FLAG', 'false')
os.environ['MEDIAPIPE_DISABLE_GPU'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['MPLCONFIGDIR'] = os.getenv('MPLCONFIGDIR', '/tmp/mpl')

import src.cvi.api.inference_service as inf
importlib.reload(inf)
run_full_cvi_pipeline = inf.run_full_cvi_pipeline

print('CFN_EMB_MODEL_PATH:', os.environ['CFN_EMB_MODEL_PATH'])
print('CFN_PROB_THRESH:', os.environ['CFN_PROB_THRESH'])
print('CFN_RATIO_THRESH:', os.environ['CFN_RATIO_THRESH'])
print('CFN_CAUSAL_THRESH:', os.environ['CFN_CAUSAL_THRESH'])
print('CFN_REQUIRE_FLAG:', os.environ['CFN_REQUIRE_FLAG'])


from pathlib import Path

manifest = PROJECT_ROOT / 'eval_manifest_balanced.tsv'
tp = tn = fp = fn = skipped = 0

for raw in manifest.read_text().splitlines():
    line = raw.strip()
    if not line:
        continue
    path_str, label_str = line.rsplit('\t', 1)
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    try:
        out = run_full_cvi_pipeline(str(p))
    except Exception as e:
        skipped += 1
        print('Skip', p, e)
        continue

    pred = int(out['video_fake'])
    label = int(label_str)
    if pred == 1 and label == 1:
        tp += 1
    elif pred == 0 and label == 0:
        tn += 1
    elif pred == 1 and label == 0:
        fp += 1
    else:
        fn += 1

total = tp + tn + fp + fn
acc = (tp + tn) / total if total else 0.0
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec = tp / (tp + fn) if (tp + fn) else 0.0
spec = tn / (tn + fp) if (tn + fp) else 0.0
bal_acc = 0.5 * (rec + spec)
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

print(f'Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} Spec={spec:.3f} BalAcc={bal_acc:.3f} F1={f1:.3f} TP={tp} TN={tn} FP={fp} FN={fn} Skipped={skipped}')


Skip /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/data/raw/fakeavceleb/FakeVideo-RealAudio/Asian (South)/men/id03344/00114_id07194_wavtolip.mp4 name 'run_full_cvi_pipeline' is not defined
Skip /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/data/raw/fakeavceleb/FakeVideo-RealAudio/Caucasian (American)/men/id01201/00028_id04034_8ZeJXbjFa_o.mp4 name 'run_full_cvi_pipeline' is not defined
Skip /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/data/raw/fakeavceleb/FakeVideo-RealAudio/Asian (South)/women/id06752/00221_id00080_wavtolip.mp4 name 'run_full_cvi_pipeline' is not defined
Skip /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/data/raw/fakeavceleb/FakeVideo-FakeAudio/Caucasian (American)/men/id00184/00241_id01239_wavtolip.mp4 name 'run_full_cvi_pipeline' is not defined
Skip /Users/venturit/Documents/GitHub/FYP/CausalX-Project/backend/data/raw/fakeavceleb/RealVideo-RealAudio/Caucasian (American)/men/id00049/00118.mp4 name 'run_full_